<a href="https://colab.research.google.com/github/ghizlane89/0__GenIA/blob/Bootcamp/W13_D3_MiniProjet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Affinage d’un modèle de Question-Réponse avec Haystack
# Challenge quotidien - Mai 2025
#
# Ce notebook montre pas à pas comment mettre en place
# un système de QA personnalisé basé sur Haystack avec un dataset au format SQuAD.

# ============================================================================
# 1. MISE EN PLACE DE L’ENVIRONNEMENT COLAB
# ============================================================================

print("🚀 Lancement du projet Fine-tuning QA avec Haystack")
print("=" * 60)

# Installation des librairies nécessaires (à activer sur Colab uniquement)
print("📦 Téléchargement et installation de Haystack + dépendances...")

# Exécuter ces lignes uniquement dans Colab (décommenter)
# !pip install --upgrade pip
# !pip install farm-haystack[inference]
# !pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

# Import des librairies Python utiles
import os
import json
import logging
import requests
from pathlib import Path
import pandas as pd

# Configuration du logger pour mieux suivre l’exécution
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

print("✅ Environnement prêt !")

# ============================================================================
# 2. PARAMÉTRAGE DE LA TÉLÉMÉTRIE
# ============================================================================

print("\n📊 Réglage de la télémétrie Haystack...")

# On désactive la collecte de métriques pour ce notebook
os.environ["HAYSTACK_TELEMETRY_ENABLED"] = "False"

print("✅ Télémétrie désactivée")


🚀 Lancement du projet Fine-tuning QA avec Haystack
📦 Téléchargement et installation de Haystack + dépendances...
✅ Environnement prêt !

📊 Réglage de la télémétrie Haystack...
✅ Télémétrie désactivée


In [7]:
# ============================================================================
# 3. INITIALISATION DU FARMREADER (versions pinnées)
# ============================================================================

print("\n🧠 Chargement du composant FARMReader...")

import torch
USE_GPU = bool(torch.cuda.is_available())

try:
    from haystack.nodes import FARMReader, BM25Retriever
    from haystack.document_stores import InMemoryDocumentStore
    try:
        from haystack import Document, Pipeline
    except ImportError:
        from haystack.schema import Document
        from haystack import Pipeline

    print("✅ Modules Haystack importés")

    try:
        print("🔄 Chargement du modèle 'deepset/roberta-base-squad2' ...")
        base_reader = FARMReader(
            model_name_or_path="deepset/roberta-base-squad2",
            use_gpu=USE_GPU,
            top_k=1,
            max_seq_len=384
        )
        print(f"✅ FARMReader initialisé (GPU={USE_GPU})")
    except Exception as e:
        print(f"⚠️ Problème de chargement (GPU={USE_GPU}) : {e}")
        print("↩️ Tentative en CPU…")
        base_reader = FARMReader(
            model_name_or_path="deepset/roberta-base-squad2",
            use_gpu=False,
            top_k=1,
            max_seq_len=384
        )
        print("✅ FARMReader initialisé en CPU")

except Exception as e:
    print(f"❌ Échec imports/chargement : {e}")
    base_reader = None

if base_reader is None:
    import pkg_resources, platform
    print(
        "Env report →",
        {"python": platform.python_version(),
         "pkgs": {d.project_name: d.version for d in pkg_resources.working_set if d.project_name in ["farm-haystack","transformers","torch"]}}
    )
    raise RuntimeError("FARMReader indisponible : vérifie les versions installées ci-dessus.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.4/764.4 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 15.4 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=136743ca0e571dabb5287c0f1819a22b6a796f606f6db9c4f432e486977a489f
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf4605

RuntimeError: Le FARMReader n'a pas pu être initialisé. Vérifiez l'installation de Haystack/PyTorch puis ré-exécutez cette cellule.

In [ ]:

# ============================================================================
# 4. TÉLÉCHARGEMENT ET PRÉPARATION DU DATASET
# ============================================================================

print("\n📂 Téléchargement et formatage du dataset SQuAD...")

# Création d’un dossier pour stocker les données
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Lien vers l’ensemble d’entraînement SQuAD 2.0
squad_url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
dataset_path = data_dir / "train-v2.0.json"

def download_squad_dataset():
    """Télécharge SQuAD 2.0 si absent en local"""
    if not dataset_path.exists():
        print("🔄 Récupération du dataset...")
        response = requests.get(squad_url)
        with open(dataset_path, "wb") as f:
            f.write(response.content)
        print("✅ Dataset téléchargé")
    else:
        print("✅ Dataset déjà disponible")

# On récupère le dataset
download_squad_dataset()

def load_squad_dataset(file_path, max_samples=1000):
    """Lit le JSON de SQuAD et extrait quelques exemples"""
    print(f"🔍 Lecture du dataset: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"📊 Détails du jeu de données :")
    print(f"   - Version: {data.get('version', 'Non spécifiée')}")
    print(f"   - Nombre d’articles: {len(data['data'])}")

    # Extraction des QA pairs
    training_examples = []
    for article in data["data"][:5]:  # on limite pour ce tuto
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:  # uniquement celles avec réponse
                    example = {
                        "question": qa["question"],
                        "context": context,
                        "id": qa["id"],
                        "answers": qa["answers"]
                    }
                    training_examples.append(example)
                    if len(training_examples) >= max_samples:
                        break
            if len(training_examples) >= max_samples:
                break
        if len(training_examples) >= max_samples:
            break

    print(f"✅ {len(training_examples)} exemples collectés")

    # Affichage d’un échantillon
    if training_examples:
        example = training_examples[0]
        print("\n📝 Exemple:")
        print(f"   Question: {example['question'][:100]}...")
        print(f"   Contexte: {example['context'][:150]}...")
        print(f"   Réponse: {example['answers'][0]['text']}")

    return training_examples

# On charge un sous-ensemble pour l’entraînement
training_data = load_squad_dataset(dataset_path, max_samples=500)

In [10]:

# ============================================================================
# 5. ENTRAÎNEMENT DU READER
# ============================================================================

print("\n🏋️ Démarrage du fine-tuning du modèle...")

# Dossier de sauvegarde du modèle affiné
model_save_dir = "fine_tuned_qa_model"
os.makedirs(model_save_dir, exist_ok=True)

# Conversion du dataset SQuAD → format attendu par Haystack
def prepare_training_data(examples):
    """Met en forme les exemples SQuAD pour Haystack"""
    train_data = []
    for example in examples:
        training_example = {
            "question": example["question"],
            "context": example["context"],
            "answer": {
                "text": example["answers"][0]["text"],
                "start": example["answers"][0]["answer_start"]
            }
        }
        train_data.append(training_example)
    return train_data

print("🔄 Conversion du dataset...")
formatted_training_data = prepare_training_data(training_data)

# Paramètres d’entraînement
training_config = {
    "data_dir": str(data_dir),
    "train_filename": "formatted_train_data.json",
    "dev_filename": None,
    "max_seq_len": 384,
    "doc_stride": 128,
    "max_query_length": 64,
    "n_epochs": 1,   # pour l’exemple : 1 epoch
    "batch_size": 8,
    "learning_rate": 3e-5,
    "save_dir": model_save_dir
}

# Sauvegarde du dataset préparé
train_file_path = data_dir / training_config["train_filename"]
with open(train_file_path, "w", encoding="utf-8") as f:
    json.dump(formatted_training_data, f, ensure_ascii=False, indent=2)

print(f"✅ Dataset formaté enregistré: {train_file_path}")

print("🔄 Simulation de l’entraînement...")
print("⚠️ Sur Colab, le vrai entraînement prendrait quelques minutes avec GPU")

"""
try:
    base_reader.train(
        data_dir=training_config["data_dir"],
        train_filename=training_config["train_filename"],
        use_gpu=True,
        n_epochs=training_config["n_epochs"],
        batch_size=training_config["batch_size"],
        learning_rate=training_config["learning_rate"],
        save_dir=training_config["save_dir"]
    )
    print("✅ Entraînement terminé")
except Exception as e:
    print(f"❌ Erreur d’entraînement: {e}")
"""

print("✅ Configuration prête (simulation dans ce script)")


🏋️ Démarrage du fine-tuning du modèle...
🔄 Conversion du dataset...
✅ Dataset formaté enregistré: data/formatted_train_data.json
🔄 Simulation de l’entraînement...
⚠️ Sur Colab, le vrai entraînement prendrait quelques minutes avec GPU
✅ Configuration prête (simulation dans ce script)


In [11]:
# ============================================================================
# 6. CHARGEMENT ET UTILISATION DU MODÈLE FINE-TUNÉ (robuste)
# ============================================================================

print("\n🔮 Chargement du modèle fine-tuné (si présent) ...")

try:
    fine_tuned_reader = FARMReader(
        model_name_or_path=model_save_dir,  # dossier du checkpoint
        use_gpu=USE_GPU,
        top_k=1,
        max_seq_len=384
    )
    print(f"✅ Modèle fine-tuné chargé depuis {model_save_dir}")
except Exception as e:
    print(f"ℹ️ Checkpoint introuvable ou invalide ({e}).")
    if 'base_reader' in globals() and base_reader is not None:
        fine_tuned_reader = base_reader
        print("➡️  Utilisation du modèle de base à la place.")
    else:
        # Ultime filet de sécurité (rarement nécessaire si la cellule 3 a bien tourné)
        print("↩️  Tentative de re-création d'un modèle de base (CPU).")
        fine_tuned_reader = FARMReader(
            model_name_or_path="deepset/roberta-base-squad2",
            use_gpu=False,
            top_k=1,
            max_seq_len=384
        )
        print("✅ Modèle de base recréé (CPU).")



🔮 Chargement du modèle fine-tuné (si présent) ...
ℹ️ Checkpoint introuvable ou invalide (Failed to import 'transformers.modeling_utils'. Run 'pip install farm-haystack[inference]'. Original error: cannot import name 'SequenceSummary' from 'transformers.modeling_utils' (/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py)).
↩️  Tentative de re-création d'un modèle de base (CPU).


ImportError: Failed to import 'transformers.modeling_utils'. Run 'pip install farm-haystack[inference]'. Original error: cannot import name 'SequenceSummary' from 'transformers.modeling_utils' (/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py)

In [12]:

# ============================================================================
# 7. TEST DU MODÈLE
# ============================================================================

print("\n🎯 Évaluation rapide...")

# Jeu de documents tests
test_documents = [
    Document(content="""L'intelligence artificielle (IA) permet aux machines de simuler
    des capacités humaines comme l’apprentissage, le langage et la vision.
    Ses usages couvrent la santé, la conduite autonome et les assistants vocaux.""", id="doc1"),
    Document(content="""Haystack est un framework open source conçu par deepset
    pour créer des moteurs de recherche basés sur les modèles de langage.
    Il s’intègre facilement à des backends classiques ou neuronaux.""", id="doc2")
]

# Questions de validation
test_questions = [
    "Qu'est-ce que l'intelligence artificielle ?",
    "Qui est à l'origine de Haystack ?",
    "Dans quels domaines l’IA est-elle appliquée ?",
    "Haystack peut-il s’intégrer avec d’autres systèmes ?"
]

def test_qa_model(reader, documents, questions):
    """Teste le modèle QA sur quelques questions"""
    print("🔍 Début des prédictions...")
    results = []
    for i, question in enumerate(questions):
        print(f"\n❓ Question {i+1}: {question}")
        try:
            prediction = reader.predict(query=question, documents=documents, top_k=1)
            if prediction["answers"]:
                answer = prediction["answers"][0]
                print(f"✅ Réponse: {answer.answer}")
                print(f"🎯 Score: {answer.score:.3f}")
                print(f"📄 Source: {answer.document_id}")
                results.append({
                    "question": question,
                    "answer": answer.answer,
                    "confidence": answer.score,
                    "document_id": answer.document_id
                })
            else:
                print("❌ Pas de réponse")
                results.append({
                    "question": question,
                    "answer": "Aucune réponse",
                    "confidence": 0.0,
                    "document_id": None
                })
        except Exception as e:
            print(f"❌ Erreur prédiction: {e}")
    return results

test_results = test_qa_model(fine_tuned_reader, test_documents, test_questions)


🎯 Évaluation rapide...


NameError: name 'fine_tuned_reader' is not defined

In [13]:

# ============================================================================
# 8. PIPELINE COMPLET (OPTIONNEL)
# ============================================================================

print("\n🔧 Construction d’un pipeline QA...")

try:
    document_store = InMemoryDocumentStore()
    document_store.write_documents(test_documents)

    retriever = BM25Retriever(document_store=document_store)

    qa_pipeline = Pipeline()
    qa_pipeline.add_node(component=retriever, name="Retriever", inputs=["Query"])
    qa_pipeline.add_node(component=fine_tuned_reader, name="Reader", inputs=["Retriever"])

    print("✅ Pipeline opérationnel")

    pipeline_question = "Quelles applications concrètes a l’IA ?"
    result = qa_pipeline.run(query=pipeline_question, params={"Retriever": {"top_k": 5}, "Reader": {"top_k": 1}})
    print(f"\n❓ Question: {pipeline_question}")
    if result["answers"]:
        answer = result["answers"][0]
        print(f"✅ Réponse pipeline: {answer.answer}")
        print(f"🎯 Score: {answer.score:.3f}")
    else:
        print("❌ Rien trouvé")
except Exception as e:
    print(f"⚠️ Pipeline non disponible: {e}")


🔧 Construction d’un pipeline QA...
⚠️ Pipeline non disponible: name 'fine_tuned_reader' is not defined


In [14]:

# ============================================================================
# 9. RAPPORT DE RÉSULTATS
# ============================================================================

print("\n📋 Résumé final:")
print("=" * 60)

results_summary = {
    "exercise": "Fine-tuning QA avec Haystack",
    "date": "2025-05-08",
    "dataset": "SQuAD 2.0 (500 exemples)",
    "base_model": "deepset/roberta-base-squad2",
    "epochs": 1,
    "questions_testées": len(test_questions),
    "réponses_valides": len([r for r in test_results if r["confidence"] > 0]),
    "score_moyen": sum([r["confidence"] for r in test_results]) / len(test_results),
    "détails": test_results
}

results_file = "qa_training_results.json"
with open(results_file, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

print("✅ Rapport sauvegardé:", results_file)



📋 Résumé final:


NameError: name 'test_results' is not defined